# Assignment 1 2AMM10 2025-2026

## Group: [Fill in your group name]
### Member 1: [Fill in your name]
### Member 2: [Fill in your name]
### Member 3: [Fill in your name]

## Task 1 

Dataset and visualization

In [1]:
import sys
print(sys.executable) 

import sys
!{sys.executable} -m pip install kagglehub

c:\Users\diego\AppData\Local\Programs\Python\Python311\python.exe



[notice] A new release of pip available: 22.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
from pathlib import Path
from torch.utils.data import Dataset
from PIL import Image
import kagglehub
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

class AppleDataset(Dataset):
    def __init__(self, transform=None, subset="train",class_subset = "main"):
        assert subset in ["train","test"]
        assert class_subset in ["main","new","all"]
        path = kagglehub.dataset_download("moltean/fruits")
        base = Path(path) / "fruits-360_original-size" / "fruits-360-original-size"
        if subset == "train":
            self.path = base / "Training"
        elif subset == "test":
            self.path = base / "Validation"
        self.transform = transform
        all_folders = sorted(os.listdir(self.path))
        self.item_folders = sorted(x for x in all_folders if x.lower().startswith("apple"))
        generator=np.random.default_rng(6)
        generator.shuffle(self.item_folders)
        if class_subset == "main":
            self.item_folders = self.item_folders[:20]
        elif class_subset == "new":
            self.item_folders = self.item_folders[20:]
        self.targets = []
        self.image_paths = []
        for i, folder in enumerate(self.item_folders):
            for img_file in sorted(os.listdir(self.path / folder)):
                if img_file.startswith("r0"):
                    if class_subset=="new":
                        self.targets.append(i+20)
                    else: 
                        self.targets.append(i)
                    self.image_paths.append(self.path / folder / img_file)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, i):
        image = Image.open(self.image_paths[i]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.targets[i]

dataset = AppleDataset()

item_dd = widgets.Dropdown(options=dataset.item_folders, description="Variety:")
frame_slider = widgets.IntSlider(value=0, min=0, max=0, description="Frame:")
output = widgets.Output()


def get_frames(folder):
    return sorted(f for f in os.listdir(dataset.path / folder)
                  if f.startswith("r0_") and f.endswith(".jpg"))

def update_slider(*_):
    frames = get_frames(item_dd.value)
    frame_slider.max = max(0, len(frames) - 1)
    frame_slider.value = min(frame_slider.value, frame_slider.max)
    show_image()

def show_image(*_):
    frames = get_frames(item_dd.value)
    if not frames or frame_slider.value >= len(frames):
        return
    with output:
        clear_output(wait=True)
        img = Image.open(dataset.path / item_dd.value / frames[frame_slider.value])
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.imshow(img)
        ax.set_title(f"{item_dd.value} | frame {frame_slider.value}")
        ax.axis("off")
        plt.tight_layout()
        plt.show()

item_dd.observe(update_slider, names="value")
frame_slider.observe(show_image, names="value")

update_slider()
display(widgets.VBox([item_dd, frame_slider, output]))

In [3]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_data = AppleDataset(subset="train", transform=train_transform)
test_data = AppleDataset(subset="test", transform=eval_transform)
support_new_data = AppleDataset(subset="train", transform=eval_transform, class_subset="new")
test_new_data = AppleDataset(subset="test", transform=eval_transform, class_subset="new")




We added some augmentation because the train set is quite small, we added very small transformation, just some horizontal flip and colorjitter.

For the neural network to be trained we think that Triplet Loss is our best option, because it is able to handle very well unseen objects and we can easily get all the pairs and labels needed. Now we need to use a Loss function that pushes the negative as far as possible from the anchor and the positive needs to be close to the anchor.

Each batch we grab P different apple varieties, and for each variety we pick K random photos.
Each batch we grab P different apple varieties, and for each variety we pick K random photos. This way, every batch always has photos of the same apple (for positives) and photos of different apples (for negatives) ready to compare against each other.

In [4]:
import numpy as np
from torch.utils.data import Sampler
from collections import defaultdict

class BalancedBatchSampler(Sampler):

    def __init__(self, dataset, P=8, K=4):
        self.P = P
        self.K = K
        
        # Build a mapping: item_label -> list of indices
        self.label_to_indices = defaultdict(list)
        for idx, label in enumerate(dataset.labels):
            self.label_to_indices[label].append(idx)
        
        # Only keep items that have at least K images
        self.valid_labels = [
            label for label, indices in self.label_to_indices.items()
            if len(indices) >= self.K
        ]
        
        assert len(self.valid_labels) >= self.P, (
            f"Not enough items with >= {K} images. "
            f"Found {len(self.valid_labels)}, need at least {P}."
        )
        
        self.n_batches = len(self.valid_labels) * K // (P * K)  # approx batches per epoch
    
    def __iter__(self):
        for _ in range(self.n_batches):
            batch_indices = []
            
            selected_labels = np.random.choice(self.valid_labels, size=self.P, replace=False)
            
            for label in selected_labels:
                indices = self.label_to_indices[label]
                sampled = np.random.choice(indices, size=self.K, replace=len(indices) < self.K)
                batch_indices.extend(sampled.tolist())
            
            yield batch_indices
    
    def __len__(self):
        return self.n_batches

In [5]:
from torch.utils.data import DataLoader

train_data.labels = [train_data[i][1] for i in range(len(train_data))]

sampler = BalancedBatchSampler(train_data, P=8, K=4)

train_loader = DataLoader(
    train_data,
    batch_sampler=sampler,
    num_workers=0,
    pin_memory=False
)

Now triplets are made, but we need to embed the pictures into small sized vectors to be able to apply Triplets Loss correctly. Resnet already contains all the architecture that we need to represent the pictures of apples as embeddings. Therefore, we will finetune resnet by removingthe classification layer and retraining the last Resnet block. We will save a lot of time and some headaches doing it this way :)

In [6]:
import torch
import torch.nn as nn
from torchvision import models

class EmbeddingNet(nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()
        backbone = models.resnet18(pretrained=True)
        
        # Remove the final classification layer
        self.encoder = nn.Sequential(*list(backbone.children())[:-1])
        
        # Add our own projection head
        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, embedding_dim),
            nn.LayerNorm(embedding_dim)
        )

    def forward(self, x):
        features = self.encoder(x)
        embeddings = self.projection(features)
        # L2 normalize so all embeddings lie on a unit sphere
        return nn.functional.normalize(embeddings, p=2, dim=1)

model = EmbeddingNet(embedding_dim=128)
print(model)

c:\Users\diego\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\diego\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


EmbeddingNet(
  (encoder): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=

We will use the triplet loss function, which looks like this triplet_loss_val = F.relu(d_ap - d_an + margin)
This loss will push to zero if the distance from postiive to anchor is really low, distances from negative to anchor is high. We are also including the semi-hard negatives because we want to find the sweet spot in between using uncertain samples, the ones that are not too far, but we also dont want to get samples too hard because the model will overfit

In [7]:
import torch
import torch.nn.functional as F

def triplet_loss(embeddings, labels, margin=0.3):

    dist_matrix = torch.cdist(embeddings, embeddings, p=2)

    loss = 0.0
    n_triplets = 0

    for i in range(len(labels)):
        anchor_label = labels[i]

        # Find positives and negatives for this anchor
        positive_mask = (labels == anchor_label)
        positive_mask[i] = False        # exclude anchor itself
        negative_mask = (labels != anchor_label)

        if not positive_mask.any():
            continue


        for pos_idx in positive_mask.nonzero(as_tuple=True)[0]:
            d_ap = dist_matrix[i, pos_idx] 

            neg_distances = dist_matrix[i][negative_mask]
            semi_hard = neg_distances[(neg_distances > d_ap) & 
                                      (neg_distances < d_ap + margin)]

            if len(semi_hard) == 0:
                # Fall back to hardest negative
                semi_hard = neg_distances.min().unsqueeze(0)

            # Use the closest semi-hard negative
            d_an = semi_hard.min()

            triplet_loss_val = F.relu(d_ap - d_an + margin)
            loss += triplet_loss_val
            n_triplets += 1

    return loss / n_triplets if n_triplets > 0 else torch.tensor(0.0)

In [8]:
import torch
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = EmbeddingNet(embedding_dim=128).to(device)

# Freeze encoder, train only last block + projection
for param in model.encoder.parameters():
    param.requires_grad = False
for param in list(model.encoder.children())[7].parameters():
    param.requires_grad = True

optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
scheduler = StepLR(optimizer, step_size=5, gamma=0.5)  # halve lr every 5 epochs

NUM_EPOCHS = 20

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    n_batches = 0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        embeddings = model(images)
        loss = triplet_loss(embeddings, labels, margin=0.3)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

        # Batch level progress
        if batch_idx % 5 == 0:
            print(f"  Epoch [{epoch+1}/{NUM_EPOCHS}] "
                  f"Batch [{batch_idx}/{len(train_loader)}] "
                  f"Loss: {loss.item():.4f}")

    avg_loss = total_loss / n_batches
    scheduler.step()

    # Epoch level progress
    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}] Complete | "
          f"Avg Loss: {avg_loss:.4f} | "
          f"LR: {scheduler.get_last_lr()[0]:.6f}\n"
          f"{'-'*50}")

Using device: cpu
  Epoch [1/20] Batch [0/2] Loss: 0.2389

Epoch [1/20] Complete | Avg Loss: 0.1951 | LR: 0.001000
--------------------------------------------------
  Epoch [2/20] Batch [0/2] Loss: 0.1670

Epoch [2/20] Complete | Avg Loss: 0.1472 | LR: 0.001000
--------------------------------------------------
  Epoch [3/20] Batch [0/2] Loss: 0.0811

Epoch [3/20] Complete | Avg Loss: 0.0665 | LR: 0.001000
--------------------------------------------------
  Epoch [4/20] Batch [0/2] Loss: 0.0440

Epoch [4/20] Complete | Avg Loss: 0.0407 | LR: 0.001000
--------------------------------------------------
  Epoch [5/20] Batch [0/2] Loss: 0.0018

Epoch [5/20] Complete | Avg Loss: 0.0169 | LR: 0.000500
--------------------------------------------------
  Epoch [6/20] Batch [0/2] Loss: 0.0207

Epoch [6/20] Complete | Avg Loss: 0.0411 | LR: 0.000500
--------------------------------------------------
  Epoch [7/20] Batch [0/2] Loss: 0.0856

Epoch [7/20] Complete | Avg Loss: 0.0778 | LR: 0.0005

First we trained the apple pics with resnet, to get some general embeddings. After that we changed the embeddings using the triplets loss function, which makes sure that similar images are embedded together. After this now we have correct embeddings for the images. Now when a new image comes, we just transform it to the same embedding space and see which is the closest item to it, we will then give this category to the new item, as simple as that. We can check how our algorithm is performing

In [9]:
import torch
import numpy as np

def extract_embeddings(model, dataset, device):
    """Extract embeddings for all images in a dataset"""
    model.eval()
    all_embeddings = []
    all_labels = []

    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            embeddings = model(images)
            all_embeddings.append(embeddings.cpu())
            all_labels.append(labels)

    return torch.cat(all_embeddings), torch.cat(all_labels)


print("Extracting training embeddings...")
train_embeddings, train_labels = extract_embeddings(model, train_data, device)

print("Extracting test embeddings...")
test_embeddings, test_labels = extract_embeddings(model, test_data, device)

print(f"Train embeddings shape: {train_embeddings.shape}")
print(f"Test embeddings shape: {test_embeddings.shape}")

Extracting training embeddings...
Extracting test embeddings...
Train embeddings shape: torch.Size([3068, 128])
Test embeddings shape: torch.Size([1538, 128])


In [ ]:
def evaluate(train_embeddings, train_labels, test_embeddings, test_labels):

    correct = 0
    total = len(test_embeddings)

    # Compute all distances between test and train embeddings
    dist_matrix = torch.cdist(test_embeddings, train_embeddings, p=2)

    nearest_indices = dist_matrix.argmin(dim=1)
    predicted_labels = train_labels[nearest_indices]

    correct = (predicted_labels == test_labels).sum().item()
    accuracy = correct / total * 100

    return accuracy, predicted_labels

accuracy, predicted_labels = evaluate(
    train_embeddings, train_labels,
    test_embeddings, test_labels
)

print(f"Test Accuracy: {accuracy:.2f}%")
print(f"Correct: {int(accuracy * len(test_labels) / 100)}/{len(test_labels)}")

Test Accuracy: 94.80%
Correct: 1458/1538


Now for the new data!

In [11]:
print("Extracting support embeddings...")
support_new_data.labels = [support_new_data[i][1] for i in range(len(support_new_data))]
support_embeddings, support_labels = extract_embeddings(model, support_new_data, device)

print("Extracting new test embeddings...")
test_new_embeddings, test_new_labels = extract_embeddings(model, test_new_data, device)

print(f"Support embeddings shape: {support_embeddings.shape}")
print(f"Test new embeddings shape: {test_new_embeddings.shape}")

accuracy_new, _ = evaluate(
    support_embeddings, support_labels,
    test_new_embeddings, test_new_labels
)

print(f"\nNew Items Test Accuracy: {accuracy_new:.2f}%")
print(f"Correct: {int(accuracy_new * len(test_new_labels) / 100)}/{len(test_new_labels)}")

Extracting support embeddings...
Extracting new test embeddings...
Support embeddings shape: torch.Size([1570, 128])
Test new embeddings shape: torch.Size([781, 128])

New Items Test Accuracy: 90.40%
Correct: 706/781


Now for both new data and already seen data we received really good results. Probably it is also associated at the preprocessing and data augmentation. The rest was as easy as following the lectures and practical exercises. We took the idea of the semi-hard samples from the practical exercises, which also helped to get the best performance from our model

## Task 2

In [ ]:
class GardenDataset(Dataset):
    def __init__(self, transform=None, class_level="item", subset="train", family_subset="main", item_subset="main"):
        assert class_level in ["item","family","both"]
        assert subset in ["train","test"]
        assert family_subset in ["main","new","all"]
        assert item_subset in ["main","new","all"]
        path = kagglehub.dataset_download("moltean/fruits")
        base = Path(path) / "fruits-360_original-size" / "fruits-360-original-size"
        if subset == "train":
            self.path = base / "Training"
        elif subset == "test":
            self.path = base / "Validation"
        self.transform = transform
        self.class_level = class_level

        canonical_items = sorted(
            d for d in os.listdir(base / "Training")
            if (base / "Training" / d).is_dir() and re.fullmatch(r'\S+ \d+', d)
        )
        item_to_family = {it: it.rsplit(' ', 1)[0] for it in canonical_items}
        canonical_families = sorted(set(item_to_family.values()))

        self.item_to_idx = {c: i for i, c in enumerate(canonical_items)}
        self.family_to_idx = {c: i for i, c in enumerate(canonical_families)}

        train_fam_to_items = defaultdict(list)
        for it in canonical_items:
            train_fam_to_items[item_to_family[it]].append(it)
        for fam in train_fam_to_items:
            train_fam_to_items[fam].sort(key=lambda x: int(x.rsplit(' ', 1)[1]))

        new_families = {fam for fam, its in train_fam_to_items.items() if len(its) == 1}
        new_items = set()
        for fam, its in train_fam_to_items.items():
            if len(its) >= 3:
                new_items.add(its[0]) 

        present = {
            d for d in os.listdir(self.path)
            if (self.path / d).is_dir() and re.fullmatch(r'\S+ \d+', d)
        }
        all_items = [it for it in canonical_items if it in present]

        if family_subset == "main":
            all_items = [it for it in all_items if item_to_family[it] not in new_families]
        elif family_subset == "new":
            all_items = [it for it in all_items if item_to_family[it] in new_families]

        if item_subset == "main":
            all_items = [it for it in all_items if it not in new_items]
        elif item_subset == "new":
            all_items = [it for it in all_items if it in new_items]

        self.items = all_items
        self.item_to_family = {it: item_to_family[it] for it in self.items}
        self.families = sorted(set(self.item_to_family.values()))
        self.new_families = new_families
        self.new_items = new_items

        # Build samples using canonical (global) indices
        self.image_paths = []
        self.targets_item = []
        self.targets_family = []
        for item in self.items:
            item_dir = self.path / item
            item_label = self.item_to_idx[item]
            family_label = self.family_to_idx[item_to_family[item]]
            for img_file in sorted(os.listdir(item_dir)):
                if img_file.endswith('.jpg'):
                    self.image_paths.append(item_dir / img_file)
                    self.targets_item.append(item_label)
                    self.targets_family.append(family_label)

        if class_level == "item":
            self.classes = self.items
            self.class_to_idx = self.item_to_idx
            self.targets = self.targets_item
        elif class_level == "family":
            self.classes = self.families
            self.class_to_idx = self.family_to_idx
            self.targets = self.targets_family

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        if self.class_level == "both":
             return image, self.targets_family[idx], self.targets_item[idx]
        return image, self.targets[idx]

    def get_items_for_family(self, family):
        return sorted(item for item, fam in self.item_to_family.items() if fam == family)

dataset = GardenDataset()

family_dd = widgets.Dropdown(options=dataset.families, description="Family:")
item_dd = widgets.Dropdown(options=dataset.get_items_for_family(dataset.families[0]), description="Item:")
frame_slider = widgets.IntSlider(value=0, min=0, max=0, description="Frame:")
output = widgets.Output()


def get_frames(item):
    return sorted(f for f in os.listdir(dataset.path / item) if f.endswith(".jpg"))

def update_items(*_):
    items = dataset.get_items_for_family(family_dd.value)
    item_dd.options = items
    item_dd.value = items[0]

def update_slider(*_):
    frames = get_frames(item_dd.value)
    frame_slider.max = max(0, len(frames) - 1)
    frame_slider.value = min(frame_slider.value, frame_slider.max)
    show_image()

def show_image(*_):
    frames = get_frames(item_dd.value)
    if not frames or frame_slider.value >= len(frames):
        return
    with output:
        clear_output(wait=True)
        img = Image.open(dataset.path / item_dd.value / frames[frame_slider.value])
        fig, ax = plt.subplots(figsize=(4, 4))
        ax.imshow(img)
        ax.set_title(f"{family_dd.value} | {item_dd.value} | frame {frame_slider.value}")
        ax.axis("off")
        plt.tight_layout()
        plt.show()

family_dd.observe(update_items, names="value")
item_dd.observe(update_slider, names="value")
frame_slider.observe(show_image, names="value")

update_slider()
display(widgets.VBox([family_dd, item_dd, frame_slider, output]))

In [ ]:
train_data = GardenDataset(subset="train", transform=transform)

# scenario 1 
test_data = GardenDataset(subset="test", transform=transform)

# scenario 2 
train_data_family = GardenDataset(subset="train", transform=transform, class_level="family")
test_data_family = GardenDataset(subset="test", transform=transform, class_level="family")

# scenario 3 
support_all_data = GardenDataset(subset="train", transform=transform, item_subset="all")
test_new_data = GardenDataset(subset="test", transform=transform, item_subset="new")

# scenario 4 
support_all_data_family = GardenDataset(subset="train", transform=transform, family_subset="all",class_level="family")
test_new_data_family = GardenDataset(subset="test", transform=transform, family_subset="new",class_level="family")

# your code here

## Task 3

In [ ]:
train_data_both = GardenDataset(class_level="both",transform=transform,subset="train",family_subset="main",item_subset="main")

test_data_both = GardenDataset(class_level="both",transform=transform,subset="test",family_subset="main",item_subset="main")

# your code here

## Task 4

In [ ]:
class BlackoutPixels:
    """Transform that randomly sets x% of pixels to black (0).
    
    Args:
        fraction: Fraction of pixels to black out (0.0 to 1.0).
    """
    def __init__(self, fraction=0.1):
        self.fraction = fraction

    def __call__(self, img):
        # img shape: (C, H, W)
        _, h, w = img.shape
        num_pixels = h * w
        num_black = int(num_pixels * self.fraction)

        # Random pixel indices to black out
        indices = torch.randperm(num_pixels)[:num_black]
        rows = indices // w
        cols = indices % w

        img = img.clone()
        img[:, rows, cols] = 0.0
        return img

def get_anomaly_dataset(fraction):
    transform = transforms.Compose([
        BlackoutPixels(fraction=fraction),
        ... # your transforms here
    ])
    return GardenDataset(subset="test", transform=transform, family_subset="main", item_subset="main")

# your code here
